# Physics Literature RAG

A RAG pipeline built over a corpus of my own physics papers to explore
how well a small quantized LLM can answer domain-specific scientific questions. Every step is explicit — no LangChain abstractions. These papers can be found here https://inspirehep.net/authors/1997829?ui-citation-summary=true$0

## Pipeline
  1. Fetch PDFs directly from arXiv
  2. Extract and clean text with PyMuPDF
  3. Chunk documents and compare three strategies
  4. Embed with `sentence-transformers/all-MiniLM-L6-v2`
  5. Index with FAISS for fast similarity search
  6. Generate answers with `Qwen2.5-3B-Instruct` (4-bit quantised)
  7. Evaluate chunking strategies on latency, source diversity, and answer quality

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu \
    transformers bitsandbytes accelerate langchain-text-splitters

## 1. Download Papers from arXiv
Downloads 8 of my own papers directly from arXiv by ID. Skips papers already cached locally. The 1-second sleep is just to avoid hammering the arXiv server

In [ ]:
import os, requests, time
from pathlib import Path

PAPERS = [
    ("2511.07514", "Quantum Calculations of the Cavity Shift in Electron g-2 Measurements"),
    ("2411.11971", "Hunting Axion Dark Matter with Anti-Ferromagnets: A Case Study with NiO"),
    ("2411.09761", "An EFT for Anisotropic Anti-Ferromagnets: Gapped Goldstones and Phase Transitions"),
    ("2409.18379", "First Principle Predictions for Cold Fermionic Gases near Criticality"),
    ("2212.10587", "Dispersion Relations for Dislocation Modes and Sensitivity to Lattice Structure"),
    ("2210.13516", "Optimal Anti-Ferromagnets for Light Dark Matter Detection"),
    ("2112.13873", "An Effective Field Theory of Magneto-Elasticity"),
    ("2103.09339", "First Principles Prediction of the Landau Parameter for Fermi Liquids"),
]

pdf_dir = Path("papers")
pdf_dir.mkdir(exist_ok=True)

for arxiv_id, title in PAPERS:
    path = pdf_dir / f"{arxiv_id}.pdf"
    r = requests.get(f"https://arxiv.org/pdf/{arxiv_id}", timeout=30)
    r.raise_for_status()
    path.write_bytes(r.content)
    print(f"  [downloaded] {arxiv_id} — {title[:55]}")
    time.sleep(1)

print(f"\nTotal papers: {len(list(pdf_dir.glob('*.pdf')))}")

## 2. Extract and Clean Text

PyMuPDF pulls raw text from each PDF page. The filtering step drops short lines                                                       
and lone numbers/symbols. These are usually headers, footers, or page numbers                                                          
  that add noise without useful content. Each paper ends up as a single string.



In [ ]:
import fitz, re

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text("text")
        # drop short lines and lone page numbers/symbols
        lines = [l for l in text.splitlines()
                 if len(l.strip()) > 20 and not re.match(r'^\s*[\d\W]{1,5}\s*$', l)]
        pages.append("\n".join(lines))
    return "\n\n".join(pages)

corpus = []  # list of dicts: {text, arxiv_id, title}
for arxiv_id, title in PAPERS:
    text = extract_text(pdf_dir / f"{arxiv_id}.pdf")
    corpus.append({"text": text, "arxiv_id": arxiv_id, "title": title})
    print(f"  {arxiv_id}: {len(text):,} chars")

print(f"\nTotal: {sum(len(d['text']) for d in corpus):,} chars")

## 3. Text Splitting
                                                                                                                                         
  Splits each paper into overlapping chunks. The overlap means consecutive chunks share some text, so context isn't completely lost at boundaries. This is important for physics where arguments and equations often span multiple sentences. I test three chunk sizes to see how the choice affects retrieval quality later.  

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_configs = {
    "small":  {"chunk_size": 256,  "chunk_overlap": 32},
    "medium": {"chunk_size": 512,  "chunk_overlap": 64},
    "large":  {"chunk_size": 1024, "chunk_overlap": 128},
}

def chunk_corpus(corpus, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # try paragraph breaks first, fall back to smaller units
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in corpus:
        for chunk in splitter.split_text(doc["text"]):
            # keep source metadata with each chunk for attribution later
            chunks.append({"text": chunk, "arxiv_id": doc["arxiv_id"], "title": doc["title"]})
    return chunks

split_chunks = {}
for name, cfg in chunk_configs.items():
    chunks = chunk_corpus(corpus, cfg["chunk_size"], cfg["chunk_overlap"])
    split_chunks[name] = chunks
    print(f"{name:6s}: {len(chunks):4d} chunks  (size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']})")

## 4. Generate Embeddings
We use `sentence-transformers/all-MiniLM-L6-v2` — a compact 384-dimensional model
optimised for semantic similarity.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

sample = embedder.encode("What is dark matter?")
print(f"Embedding dimension: {len(sample)}")

## 5. Build FAISS Indexes
Encodes all chunks into 384-dimensional vectors and loads them into a FAISS index.
  I use L2 normalization so that inner product search gives cosine similarity.IndexFlatIP does exact brute-force search, which is fine
  for a few hundred chunks but would need to be swapped out at larger scale. One index is built for each chunking configuration.

In [ ]:
import faiss

def build_index(chunks, embedder):
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True,
                                  convert_to_numpy=True).astype("float32")
    # normalize so inner product equals cosine similarity
    faiss.normalize_L2(embeddings)
    # IndexFlatIP does exact search — fine at this scale
    index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product = cosine after normalisation
    index.add(embeddings)
    return index, embeddings

indexes = {}
for name, chunks in split_chunks.items():
    print(f"\nIndexing '{name}' chunks...")
    t0 = time.time()
    index, embeddings = build_index(chunks, embedder)
    indexes[name] = {"index": index, "embeddings": embeddings}
    print(f"  {len(chunks)} vectors indexed in {time.time() - t0:.1f}s")

## 6. Retrieval Function
Embeds the query using the same model as the chunks, normalizes it,
  and searches the FAISS index for the top-k most similar chunks. The sanity check verifies that the retrieval is working that is the returned                                                               chunks should come from the antiferromagnet papers.

In [ ]:
def retrieve(query, config_name, k=4):
    chunks = split_chunks[config_name]
    index  = indexes[config_name]["index"]
    # embed the query the same way as the chunks
    q_emb  = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, ids = index.search(q_emb, k)
    # return (chunk dict, similarity score) pairs
    return [(chunks[i], float(scores[0][j])) for j, i in enumerate(ids[0])]

# sanity check — should return chunks from the antiferromagnet papers
results = retrieve("What makes antiferromagnets good for dark matter detection?", "large")
for chunk, score in results:
    print(f"[{score:.3f}] ({chunk['arxiv_id']}) {chunk['text'][:120]}...\n")

## 7. Load Language Model
Loads Qwen2.5-3B-Instruct in 4-bit precision using BitsAndBytes.                                                                      
  I picked this model because it runs on a free Colab T4 GPU without an API key, and follows instructions reliably enough for grounded Q&A.                                                                     Greedy decoding keeps answers deterministic that is same query always gives the same answer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, GenerationConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization — reduces memory by ~8x so it fits on a free Colab T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, # extra compression on the quantization constants
)

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)


model.generation_config = GenerationConfig(
      max_new_tokens=512,
      do_sample=False,
  )

generator = pipeline(
      "text-generation",
      model=model,
      tokenizer=tokenizer,
      return_full_text=False,
  )

print("Model ready.")

## 8. RAG Pipeline
Combines retrieval and generation into a single function.
  The prompt explicitly instructs the model to answer only from the provided context and this is what keeps it grounded and prevents hallucination.                                                                              
  Each chunk is prefixed with its arXiv ID so the model can attribute claims to sources.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

model.generation_config.max_length = None

def rag_query(query, config_name="medium", k=4):
    retrieved = retrieve(query, config_name, k=k)
    # prefix each chunk with its arXiv ID so the model knows where it came from
    context = "\n\n".join(f"[{c['arxiv_id']}] {c['text']}" for c, _ in retrieved)
    sources  = list({c["title"] for c, _ in retrieved})

    prompt = (
        f"You are a helpful scientific assistant. Answer the question using only "
        f"the context provided. If the answer is not in the context, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )

    answer = generator(prompt)[0]["generated_text"].strip()
    return {"answer": answer, "sources": sources}

# demo query — tests the full pipeline end to end
result = rag_query("What makes antiferromagnets good candidates for dark matter detection?")
print("Answer:", result["answer"])
print("\nSources:")
for s in result["sources"]: print(f"  - {s}")

## 9. Query the System

In [ ]:
queries = [
    "What makes antiferromagnets good candidates for dark matter detection?",
    "How does the EFT of magneto-elasticity couple phonons and magnons?",
    "What are the main predictions for cold fermionic gases near the unitary limit?",
    "What is the significance of the cavity shift in electron g-2 measurements?",
    "How does lattice anisotropy affect dislocation dispersion relations?",
]

for q in queries:
    r = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {r['answer']}")
    print(f"Sources: {[s[:50] for s in r['sources']]}\n")

## 10. Evaluate Chunking Strategies
Runs all 6 queries through all 3 chunking configurations and records:
  - latency: total time for retrieval + generation
  - unique_papers: how many distinct papers appear in the top-4 retrieved chunks
  - answer_length: word count of the generated answer

  The goal is to see whether chunk size affects what the model retrieves and
  how it responds, not just whether the answers are correct.

In [ ]:
import pandas as pd

model.generation_config.max_length = None

eval_queries = [
    "What makes antiferromagnets good candidates for dark matter detection?",
    "How does the EFT of magneto-elasticity couple phonons and magnons?",
    "What are the predictions for cold fermionic gases near the unitary limit?",
    "How does lattice anisotropy affect dislocation dispersion relations?",
    "What is the optimal antiferromagnet for sub-GeV dark matter detection?",
    "What is the role of anomaly matching in predicting thermodynamic properties?",
]

records = []
for config in ["small", "medium", "large"]:
    for query in eval_queries:
        t0 = time.time()
        result = rag_query(query, config_name=config)
        latency = time.time() - t0
        # retrieve again to get scores — rag_query doesn't expose them
        retrieved = retrieve(query, config, k=4)
        records.append({
            "chunk_config": config,
            "query": query[:50],
            "latency_s": round(latency, 2),
            "unique_papers": len({c["arxiv_id"] for c, _ in retrieved}),
            "answer_length": len(result["answer"].split()),
        })

df = pd.DataFrame(records)
print(df.groupby("chunk_config")[["latency_s", "unique_papers", "answer_length"]].mean().round(2))

## 11. Visualize Results
                                                                                                                                          
  Plots the three evaluation metrics side by side for each chunking configuration. Saved to chunking_evaluation.png.

In [ ]:
import matplotlib.pyplot as plt

metrics = ["latency_s", "unique_papers", "answer_length"]
titles  = ["Avg Latency (s)", "Avg Unique Papers Retrieved", "Avg Answer Length (words)"]
summary = df.groupby("chunk_config")[metrics].mean()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, metric, title in zip(axes, metrics, titles):
    summary[metric].plot(kind="bar", ax=ax, color=colors, edgecolor="white", width=0.6)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Chunk Configuration")
    ax.set_xticklabels(summary.index, rotation=0)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("RAG Evaluation: Chunking Strategy Comparison (8 physics papers)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("chunking_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Corpus | 8 arXiv physics papers | Domain-specific, fetched automatically |
| Embeddings | `all-MiniLM-L6-v2` | Compact 384-dim model, strong semantic similarity |
| Vector Store | FAISS (IndexFlatIP + L2 norm = cosine) | Production-grade ANN search, CPU-friendly |
| LLM | `Qwen2.5-3B-Instruct` (4-bit) | Current, runs on free T4, no API key required |
| Chunking | small / medium / large | Medium (512 tokens) balances precision and context |

### Key Observations
- **Small chunks** improve precision for narrow factual queries but lose equation and argument context
- **Large chunks** retrieve broader context but dilute relevance scores for specific questions
- **Medium chunks** (512 tokens, 64 overlap) offer the best trade-off across the evaluated query set

### Next Steps
- Add cross-encoder reranking to improve retrieved chunk ordering
- Experiment with hybrid BM25 + dense retrieval for better coverage of technical terminology
- Extend corpus with citing and cited papers from INSPIRE HEP